In [ ]:
# Copyright 2025 DeepMind Technologies Limited. All Rights Reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

[![Colab で開く](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/google-gemini/genai-processors/blob/main/notebooks/create_your_own_processor.ipynb)


# 独自の Processor を作る

このセクションでは、独自の Processor を作成する手順を段階的に解説します。

## 1. 🛠️ セットアップ

まず、GenAI Processors ライブラリをインストールします:


In [ ]:
!pip install genai-processors

### API キー

GenAI のモデルプロセッサを使用するには API キーが必要です。まだお持ちでない場合は Google AI Studio から取得し、Colab のシークレットとしてインポート（推奨）するか、以下で直接設定してください。


In [ ]:
from google.colab import userdata

API_KEY = userdata.get('GOOGLE_API_KEY')

## 2. 🎯 実装するプロセッサの種類を決める

GenAI Processors には主に 2 種類のプロセッサがあります:

1. 入力の `ProcessorPart` ストリームを順序通りに処理する標準的な `Processor`。`Processor` は次のインターフェースを実装する必要があります:
>   ```python
>     @abc.abstractmethod
>     async def call(
>          self, content: AsyncIterable[ProcessorPart]
>     ) -> AsyncIterable[ProcessorPartTypes]:
>     ...
>   ```
2. 各 `ProcessorPart` を独立かつ並行に処理する `PartProcessor`。入力引数が単一の `ProcessorPart` になる点を除き、似たインターフェースを実装します:
>   ```python
>     @abc.abstractmethod
>     async def call(
>          self, content: ProcessorPart
>     ) -> AsyncIterable[ProcessorPartTypes]:
>     ...
>   ```

どちらのプロセッサも `ProcessorPartTypes` を出力します。これは包括的な型で、`string`、`PIL.Image.Image`、`genai_part`、`ProcessorPart` を含みます。`genai_part` や `ProcessorPart` 以外を返した場合は、ライブラリが自動的に `ProcessorPart` にラップします。その際、`role` は `user` とみなし、オブジェクトから型を推論します（例: 文字列はテキスト、`PIL.Image.Image` は画像）。注意点として、raw bytes は mimetype を推論できないため、Processor/PartProcessor から直接返せません。適切な mimetype を指定した `ProcessorPart` に包んで返してください。

`PartProcessor` は `to_processor()` 関数で `Processor` に変換できます。内部的には、`content: AsyncIterable[ProcessorPart]` の各要素に対して PartProcessor が並行に適用されます。これにより、標準の Processor 実装より短時間で計算が終わることがあり、可能な限り PartProcessor の実装を優先するのが望ましいです（典型的には、`content` ストリーム内のアイテム間で計算順序が重要でない場合）。

計算順序が重要な場合（例えば、入力ストリームからテキストをバッファし、正規表現をチェックする Processor など）には、`Processor` を実装してください。

Processor と PartProcessor の計算時間の違いを見るために、次の例を考えてみましょう。時間計測の収集には 20〜30 秒ほどかかります。


In [ ]:
import asyncio
from typing import AsyncIterable
from genai_processors import content_api
from genai_processors import processor
from genai_processors import streams
import nest_asyncio

nest_asyncio.apply()  # Needed to run async loops in Colab


@processor.processor_function
async def upper_case_processor(
    content: AsyncIterable[content_api.ProcessorPart],
) -> AsyncIterable[content_api.ProcessorPartTypes]:
  async for part in content:
    if content_api.is_text(part.mimetype):
      yield part.text.upper()
    else:
      yield part
    # Sleep a bit to simulate more compute intensive task
    await asyncio.sleep(0.001)


@processor.part_processor_function
async def upper_case_part_processor(
    part: content_api.ProcessorPart,
) -> AsyncIterable[content_api.ProcessorPartTypes]:
  # The code below is the same block as the `async for` block in the function
  # above.
  if content_api.is_text(part.mimetype):
    yield part.text.upper()
  else:
    yield part
  # Sleep a bit to simulate more compute intensive task
  await asyncio.sleep(0.001)


async def load_test(processor: processor.Processor):
  input_stream = streams.stream_content(["hello"] * 1000)
  async for _ in processor(input_stream):
    pass


print("time with Processor:")
%timeit asyncio.run(load_test(upper_case_processor))
print("time with PartProcessor:")
%timeit asyncio.run(load_test(upper_case_part_processor.to_processor()))

PartProcessor は、アプリケーションでそのまま使われることはあまりありません。通常は、複雑な Processor を分解して得られる下位処理ユニットとして用い、それらを `+`（連結）や `//`（並列）の演算子で組み合わせます。単体で利用する場合は、`to_processor()` で通常の Processor に変換して使うことが重要です。これを忘れると例外になる可能性が高いでしょう。

## 3. 🏗️ クラスとして実装するか、関数として実装するか

これまでの例では、`@processor.processor_function` デコレータで関数をラップしてプロセッサを定義しました。Processor にパラメータがある場合は、クラスとして定義すると便利なことがあります。その場合は `processor.Processor` もしくは `processor.PartProcessor` を拡張し、`call` メソッドを実装します:


In [ ]:
from genai_processors import processor


class PreambleProcessor(processor.Processor):
  """Adds a preamble to the content."""

  def __init__(self, preamble: content_api.ProcessorContent):
    self._preamble = preamble

  async def call(
      self,
      content: AsyncIterable[content_api.ProcessorPart],
  ) -> AsyncIterable[content_api.ProcessorPartTypes]:
    for part in self._preamble:
      yield part
    async for part in content:
      yield part


p = PreambleProcessor([
    "Instruction manual: RP-60 is a rotary retro phone. To dial a number, put",
    " your finger in a hole opposite the desired digit and rotate the disk",
    " clockwise...",
])
input_stream = streams.stream_content(["Where are the buttons?"])

async for part in p(input_stream):
  print(part)

## 4. ⚙️ 状態管理に関する注意

内部状態が必要な場合は、以下のように `call()` メソッド内で管理するのがベストプラクティスです:


In [ ]:
async def call(
    self,
    content: AsyncIterable[content_api.ProcessorPart],
) -> AsyncIterable[content_api.ProcessorPartTypes]:
  # define your state variables
  state = ...

  for part in self._preamble:
    # Update your state variable
    state.update()

同じ Processor を異なる入力ストリームに対して呼び出しても、状態変数に副作用を生じさせないで済みます。

もしクラスレベルで状態変数を作成する必要がある場合は、同一の `call()` 実行中にその状態変数へ 2 度アクセスされたら例外を投げる、という運用が推奨です。これは状態が 2 度の実行にまたがってしまうことを示します。以下を参照してください。


In [ ]:
class MyProcessor(processor.Processor):

  def __init__(self):
    self._queue: asyncio.Queue | None = None

  async def call(
      self,
      content: AsyncIterable[content_api.ProcessorPart],
  ) -> AsyncIterable[content_api.ProcessorPartTypes]:
    if self._queue is not None:
      raise ValueError("My Processor can only be called once.")
    self._queue = asyncio.Queue()
    try:
      ...
      async for part in content:
        ...
    finally:
      self._queue = None

このパターンにより、同一時点で 1 つの入力ストリームに対してのみ Processor を呼び出せるようになり、共有状態による予期せぬ副作用を防げます。


## 5. ⚡ Processor 内でのタスク生成

場合によっては、Processor がデータを並行処理するために asyncio タスクを生成する必要があります。GenAI Processors ライブラリは、ジェネレーターや例外処理と整合する形でタスクを管理する `processor.create_task` を提供しています。タスクの作成にはこの関数の利用を強く推奨します。内部では TaskGroup に似たコンテキストマネージャを用い、タスクのキャンセルや例外を適切に扱います。


In [ ]:
from genai_processors import processor


class TeaProcessor(processor.Processor):

  async def _wait_for_tea_to_brew(self):
    await asyncio.sleep(1)
    print("Your acme super-express tea is ready!")

  async def call(
      self, content: AsyncIterable[content_api.ProcessorPart]
  ) -> AsyncIterable[content_api.ProcessorPartTypes]:
    yield "Please have a tea while we process your request"
    tea_task = processor.create_task(self._wait_for_tea_to_brew())
    async for part in content:
      if content_api.is_text(part.mimetype):
        print(part.text)
    await tea_task


input_stream = streams.stream_content(
    ["Actually ", " I wanted cofee."], with_delay_sec=0.6
)
async for _ in TeaProcessor()(input_stream):
  pass

`call()` メソッド内の `async for` ループの本体は、ブロッキングしないようにしてください。長時間かかる処理がある場合は、それを asyncio タスクに包み、必要に応じてイベントループが他のタスクに切り替えられるようにするのが強く推奨されます。

`processor.create_task` で実行される `async def` 関数を用いれば自然にそうなります。とはいえ、長時間処理が同期関数として実装されている場合もあります。その場合は `asyncio.to_thread()` を使い、asyncio が他タスクに切り替えられるようにしてください。


In [ ]:
import time


class TeaProcessor(processor.Processor):

  def _prepare_tea(self):
    # long running operation - sync mode.
    print("Brewing tea...")
    time.sleep(1)

  async def _wait_for_tea_to_brew(self):
    # sync method with a long running operation
    await asyncio.to_thread(self._prepare_tea)
    print("Your acme super-express tea is ready!")

  async def call(
      self, content: AsyncIterable[content_api.ProcessorPart]
  ) -> AsyncIterable[content_api.ProcessorPartTypes]:
    yield "Please have a tea while we process your request"
    tea_task = processor.create_task(self._wait_for_tea_to_brew())
    async for part in content:
      if content_api.is_text(part.mimetype):
        print(part.text)
    await tea_task


input_stream = streams.stream_content(
    ["Actually ", " I wanted cofee."], with_delay_sec=0.6
)
async for _ in TeaProcessor()(input_stream):
  pass

## 6. 🔗 Processor を組み合わせる

Processor を作るときは、全体の計算を複数の小さな処理に分割できることがよくあります。それぞれを別々の Processor にし、`+` 演算子で連結すると、前の Processor の出力を次の Processor が受け取るチェーンを構築できます。PartProcessor が含まれる場合は、入力の各 Part を並行に処理しつつ出力順序を保持できます。

加えて、PartProcessor は並列演算子 `//` をサポートしています。並列 PartProcessor は同じ入力チャンクに対して実行されるため、チェーン内の前段 Processor の完了を待たずに Part を出力できます。これは、互いに独立な変換（例: 異なるドキュメントタイプ（PDF/DOC/PPT）をモデルが理解しやすい表現に変換する PartProcessor など）に便利です。出力される `ProcessorPart` の順序は、入力チャンクの順序に従います。同一の入力パートに対して複数の Processor が出力を生成する場合、その出力順序はチェーン内の Processor の並び順に従います。

また、`ProcessorPart` ストリームを分割・結合するユーティリティも提供しています。`processor.parallel_concat()` は複数プロセッサの出力ストリームを連結し、`streams.split` はストリームを 2 本以上に分割します。分割後のストリームは複数の Processor で並列処理できます。

例として、前置きテキストを付けてからモデルに送信し、その結果を大文字化する複合 Processor を定義してみましょう:


In [ ]:
from genai_processors.core import genai_model
from genai_processors.core import preamble
from google.colab import userdata
from google.genai import types as genai_types


class UpperGenAI(processor.Processor):

  def __init__(self):
    self._preamble = preamble.Preamble(content=['what is the definition of: '])
    self._model = genai_model.GenaiModel(
        # Use your API KEY here
        api_key=userdata.get('GOOGLE_API_KEY'),
        model_name='gemini-2.0-flash',
        generate_content_config=genai_types.GenerateContentConfig(
            temperature=0.7
        ),
    )
    self._post_processing = upper_case_processor

  async def call(
      self, content: AsyncIterable[content_api.ProcessorPart]
  ) -> AsyncIterable[content_api.ProcessorPartTypes]:
    p = self._preamble + self._model + self._post_processing
    async for part in p(content):
      yield part


input_stream = streams.stream_content(['processor'])
async for part in UpperGenAI()(input_stream):
  print(part.text)

Processor は PartProcessor とチェーンで組み合わせられ、両者間の変換はライブラリが処理します。チェーンが PartProcessor のみで構成される場合、そのチェーン自体も PartProcessor のままで、前述の効率性のメリットを維持します。したがって、可能なら途中に Processor を挟まず、PartProcessor 同士で連結することを推奨します。

## 7. 🚧 デバッグとテスト

Processor を書いたら、内部で何が起きているかを観察したくなるでしょう。`debug` ライブラリにはパイプライン内の `ProcessorPart` を記録するロギング用プロセッサがいくつか用意されています。以下のコードは `debug.print_stream()` プロセッサを使って、2 つの Processor の間でストリームの内容を出力する例です。`print_stream` は各 `ProcessorPart` を変更せずに印字し、その後そのまま返します。Colab 以外の環境では、同様に動作する `debug.log_stream()`（print ではなく logging を使用）も利用できます。


In [ ]:
from genai_processors import debug


class UpperGenAIWithLogs(processor.Processor):

  def __init__(self):
    self._preamble = preamble.Preamble(
        content=['In two sentences, what is the definition of: ']
    )
    self._model = genai_model.GenaiModel(
        api_key=API_KEY,
        model_name='gemini-2.0-flash',
        generate_content_config=genai_types.GenerateContentConfig(
            temperature=0.7
        ),
    )
    self._post_processing = upper_case_processor

  async def call(
      self, content: AsyncIterable[content_api.ProcessorPart]
  ) -> AsyncIterable[content_api.ProcessorPartTypes]:
    p = (
        self._preamble
        # Intercept any ProcessorPart in this chain and prints it.
        # The input arg is a label indicating where the log is captured.
        + debug.print_stream('Before Model')
        + self._model
        # Intercept any ProcessorPart in this chain and prints it.
        # The input arg is a label indicating where the log is captured.
        + debug.print_stream('After Model')
        + self._post_processing
    )
    async for part in p(content):
      yield part


input_stream = streams.stream_content(['processor'])
async for part in UpperGenAIWithLogs()(input_stream):
  print(part.text)

出力を見ると、`ProcessorPart` が相互に入り組んでいるのが分かります。これは双方向ストリーミングの典型で、最後の `self._post_processing` Processor が入力ストリームをすべて消費し終える前に各 `ProcessorPart` を逐次処理し、出力を生成している状況です。このデバッグ手法を使う場合は、"After Model" や "Before Model" のようなラベルを付け、ログをそれでフィルタすると役立つことがあります。

Processor のテストは、`IsolatedAsyncioTestCase` を使い、通常の `async for` ループで結果を集めるだけで簡単に行えます:

```python
class TestUpperCaseProcessor(unittest.IsolatedAsyncioTestCase):

  async def test_to_upper_case_ok(self):
    expected = "HELLO WORLD!"
    input_stream = streams.stream_content(["hello ", "world!"])
    actual = content_api.ProcessorContent()
    async for part in upper_case_processor(input_stream):
      actual += part
    # Only collect the processor output from the default substream to filter out
    # any status or debug statements.
    self.assertEqual(actual.as_text(substream_name=""), expected)
```

同じテストは、`processor.apply_sync` メソッドを使って同期モードでも書けます:


In [ ]:
import unittest


class TestUpperCaseProcessor(unittest.TestCase):

  def test_to_upper_case_ok(self):
    expected = "HELLO WORLD!"
    actual = content_api.ProcessorContent(
        processor.apply_sync(upper_case_processor, ["hello ", "world!"])
    )
    self.assertEqual(actual.as_text(substream_name=""), expected)


if __name__ == "__main__":
  unittest.main(argv=["first-arg-is-ignored"], exit=False)

## 8. ➡️ 次のステップ

このチュートリアルでは、`Processor` と `PartProcessor` の作成方法、およびユースケースに応じたクラスの選び方を説明しました。

続けて学ぶには、
[live processor intro](https://colab.research.google.com/github/google-gemini/genai-processors/blob/main/notebooks/live_processor_intro.ipynb)
のノートブックで、Live API を使ったリアルタイムプロセッサの作成に踏み込んでみてください。
